In [0]:
# Retail Sales Lakehouse - Bronze Layer
# Source: Unity Catalog Managed Volume

source_path = "/Volumes/workspace/default/retail_sales_volume"

display(dbutils.fs.ls(source_path))

In [0]:
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{source_path}/customers.csv")
)

display(customers_df)

In [0]:
customers_df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True)
])

In [0]:
customers_df = (
    spark.read
    .option("header", "true")
    .schema(customers_schema)
    .csv(f"{source_path}/customers.csv")
)

display(customers_df)

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_customers")

In [0]:
bronze_customers_df = spark.table("workspace.default.bronze_customers")

display(bronze_customers_df)

In [0]:
bronze_customers_df = spark.table("workspace.default.bronze_customers")

display(bronze_customers_df)

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("unit_price", DoubleType(), True)
])

In [0]:
products_df = (
    spark.read
    .option("header", "true")
    .schema(products_schema)
    .csv(f"{source_path}/products.csv")
)

display(products_df)

In [0]:
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_products")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", DateType(), True),
    StructField("order_status", StringType(), True),
    StructField("sales_channel", StringType(), True)
])

In [0]:
orders_df = (
    spark.read
    .option("header", "true")
    .schema(orders_schema)
    .csv(f"{source_path}/orders.csv")
)

display(orders_df)

In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_orders")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

order_items_schema = StructType([
    StructField("order_item_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True)
])

In [0]:
order_items_df = (
    spark.read
    .option("header", "true")
    .schema(order_items_schema)
    .csv(f"{source_path}/order_items.csv")
)

display(order_items_df)

In [0]:
order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_order_items")

In [0]:
tables = [
    "workspace.default.bronze_customers",
    "workspace.default.bronze_products",
    "workspace.default.bronze_orders",
    "workspace.default.bronze_order_items"
]

for table in tables:
    df = spark.table(table)
    print(f"{table} -> {df.count()} rows")